# `04_load_ca_changes.ipynb` — Load Ken's CA Changes XLSX into normalized CSVs

Reads `data/qa_data/CA Changes breakdown.xlsx` (Ken's master record of QA corrections to the cumulative accounting) and emits the normalized rows that the proposed `ParcelDevelopmentChangeEvent` + `QaCorrectionDetail` schema (in `erd/target_schema.md`) would hold.

Outputs (all in `data/qa_data/`):
- **`qa_change_events.csv`** — parent rows (subset of `ParcelDevelopmentChangeEvent` columns)
- **`qa_correction_detail.csv`** — sidecar rows (mirrors `QaCorrectionDetail`)
- **`qa_load_orphans.csv`** — rows that failed validation, with reason

Kernel: `arcgispro-py3`. Read-only against the XLSX.

**Cadence**: re-run when Ken updates the XLSX or after a future big-sweep campaign. Idempotent when `SourceFileSnapshot` matches — the eventual DB load UPSERTs on `(RawAPN, ReportingYear, SourceFileSnapshot)`.

## 1. Imports + paths

In [1]:
import datetime as dt
from pathlib import Path
import sys
import pandas as pd

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
QA_DIR = REPO_ROOT / 'data' / 'qa_data'
QA_DIR.mkdir(parents=True, exist_ok=True)

# Make the ETL package importable so canonical_apn can come from utils.
sys.path.insert(0, str(REPO_ROOT))
from parcel_development_history_etl.utils import canonical_apn  # noqa: E402

# Source XLSX: prefer the committed copy in from_ken/, fall back to data/qa_data/.
_CANDIDATES = [REPO_ROOT / 'from_ken' / 'CA Changes breakdown.xlsx',
               QA_DIR / 'CA Changes breakdown.xlsx']
INPUT_XLSX = next((p for p in _CANDIDATES if p.exists()), None)
if INPUT_XLSX is None:
    raise FileNotFoundError(
        f'CA Changes breakdown.xlsx not found in any of: {[str(p) for p in _CANDIDATES]}'
    )

OUT_EVENTS  = QA_DIR / 'qa_change_events.csv'
OUT_DETAIL  = QA_DIR / 'qa_correction_detail.csv'
OUT_ORPHANS = QA_DIR / 'qa_load_orphans.csv'

SOURCE_SNAPSHOT = f"{INPUT_XLSX.name}@{dt.datetime.fromtimestamp(INPUT_XLSX.stat().st_mtime).strftime('%Y-%m-%d')}"
LOADED_AT = dt.datetime.now(dt.timezone.utc).isoformat()

print(f'INPUT:  {INPUT_XLSX}')
print(f'SOURCE_SNAPSHOT: {SOURCE_SNAPSHOT}')
print(f'LOADED_AT:       {LOADED_AT}')

INPUT:  C:\Users\mbindl\Documents\GitHub\Reporting\from_ken\CA Changes breakdown.xlsx
SOURCE_SNAPSHOT: CA Changes breakdown.xlsx@2026-04-30
LOADED_AT:       2026-05-04T07:57:36.418395+00:00


## 2. Read Sheet2 — TWO controlled vocabularies

Sheet2 holds **two** controlled vocabularies, one per reporting cycle:

- **Vocab A (2026 cycle)** — pivot-table block at rows 3–12, used for the `2026 Changes Reason` column on Sheet1. 9 categories.
- **Vocab B (2023 cycle)** — narrative block at rows 22–26, used for the `2023 Summary Reason` column on Sheet1. 5 categories.

Sheet1's actual category strings often paraphrase these labels rather than match them exactly (Ken's wording varies). The loader records the raw value verbatim and stores it in `CorrectionCategory`; orphan list captures the unmatched ones for downstream mapping.

In [2]:
vocab_raw = pd.read_excel(INPUT_XLSX, sheet_name='Sheet2', header=None)

# Vocab A: 2026 cycle categories (rows 3..12, first column)
VOCAB_2026 = set(
    str(v).strip() for v in vocab_raw.iloc[3:13, 0].dropna()
    if isinstance(v, str) and v.strip() and v.strip().lower() != '(blank)'
)
# Vocab B: 2023 cycle categories (rows 22..26, first column)
VOCAB_2023 = set(
    str(v).strip() for v in vocab_raw.iloc[22:27, 0].dropna()
    if isinstance(v, str) and v.strip()
)

print(f'Vocab 2026 cycle: {len(VOCAB_2026)} categories')
for c in sorted(VOCAB_2026): print(f'  - {c}')
print()
print(f'Vocab 2023 cycle: {len(VOCAB_2023)} categories')
for c in sorted(VOCAB_2023): print(f'  - {c}')

Vocab 2026 cycle: 9 categories
  - 2026 Additional Corrections to Previously Reported - Unit(s) Previously Not Counted
  - Additional Unit(s) Removed in 2026
  - Correction to Prior Analysis - Additional Unit(s) Removed in 2026
  - No Update Required
  - None - 2023 Update Correct
  - Over-Correction in 2023; Unit(s) Removed in 2026
  - Under-Correction in 2023; Unit(s) added
  - Unit(s) Incorrectly Removed in 2023 - Unit(s) Added Back in 2026
  - Unit(s) Not Previously Counted - Added in 2026

Vocab 2023 cycle: 5 categories
  - Commercial/Tourist Parcels, Not Residential, Units Removed
  - HOA/Condo Common Area/Mobile Home Parcel Corrections
  - Units Added based on Verified County/TRPA Records or GIS
  - Units Removed Based on County/TRPA Records or GIS
  - Vacant Parcels, Units Removed


## 3. Read Sheet1 — the per-APN correction log

44,371 rows. Columns observed (note the trailing space on `2026 Changes Reason `):

- `APN`
- `2023 Updates`         — quantity delta applied during the 2023 big sweep
- `2023 Update Reason`   — free-text rationale for the 2023 change
- `2023 Summary Reason`  — categorical label for the 2023 change
- `2026 Changes`         — quantity delta applied during the 2026 big sweep
- `2026 Detailed`        — free-text rationale for the 2026 change
- `2026 Changes Reason ` — categorical label for the 2026 change (controlled by Sheet2)
- `2023 CA Report Change Reason` / `2023 CA Report Change` — historical reference

In [3]:
df = pd.read_excel(INPUT_XLSX, sheet_name='Sheet1')
# Strip whitespace from column names (handles 'Reason ' trailing space)
df.columns = df.columns.str.strip()
print(f'Sheet1 rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(5)

Sheet1 rows: 44,371
Columns: ['APN', '2023 Updates', '2023 Update Reason', '2023 Summary Reason', '2026 Changes', '2026 Detailed', '2026 Changes Reason', '2023 CA Report Change Reason', '2023 CA Report Change', 'Unnamed: 9', nan, 'Unnamed: 11', 'Unnamed: 12']


,APN,2023 Updates,2023 Update Reason,2023 Summary Reason,2026 Changes,2026 Detailed,2026 Changes Reason,2023 CA Report Change Reason,2023 CA Report Change,Unnamed: 9,NaN,Unnamed: 11,Unnamed: 12
0,007-011-01,1.0,Unit(s) not previously counted. Constructed in...,Unit(s) not previously counted. Constructed in...,0.0,NaN,None - 2023 Update Correct,Unit not Previously Counted,1.0,NaN,NaN,NaN,NaN
1,007-011-023,-1.0,Parcel Number Only,Correction Based on County Data,0.0,NaN,None - 2023 Update Correct,NaN,NaN,NaN,007-011-,23.0,007-011-23
2,007-011-23,1.0,Parcel Number Only,Correction Based on County Data,2.0,Unit(s) not previously counted. Constructed in...,Under-Correction in 2023; Unit(s) added,NaN,NaN,NaN,007-011-,23.0,007-011-023
3,014-021-01,0.0,NaN,NaN,-1.0,Vacant - zero out,Additional Unit(s) Removed in 2026,NaN,NaN,NaN,014-021-,1.0,014-021-001
4,014-021-11,0.0,NaN,NaN,0.0,NaN,No Update Required,NaN,NaN,NaN,014-021-,11.0,014-021-011


## 4. APN canonicalization (sanity check)

Imported above from `parcel_development_history_etl.utils.canonical_apn`. The 2018 leading-zero APN reformat affected **multiple counties**, not just El Dorado. Pattern observed in Ken's data:

- `015-331-04`  (2-digit third segment)  ↔  `015-331-004`  (3-digit, post-2018)
- `007-011-23`  ↔  `007-011-023`
- `023-111-37`  ↔  `023-111-037`

Canonical form: pad the third segment to 3 digits when the APN matches `NNN-NNN-NN(N)`. Other formats (e.g., Douglas County's longer `1418-03-301-010`) pass through unchanged.

In [4]:
# Spot check
samples = ['015-331-04', '015-331-004', '007-011-23', '007-011-023',
           '023-111-37', '048-041-15', '1418-03-301-010', None, '']
for s in samples:
    print(f'  {repr(s):25} -> {repr(canonical_apn(s))}')

  '015-331-04'              -> '015-331-004'
  '015-331-004'             -> '015-331-004'
  '007-011-23'              -> '007-011-023'
  '007-011-023'             -> '007-011-023'
  '023-111-37'              -> '023-111-037'
  '048-041-15'              -> '048-041-015'
  '1418-03-301-010'         -> '1418-03-301-010'
  None                      -> None
  ''                        -> None


## 5. Transform Sheet1 rows → events + sidecar rows

For each Sheet1 row, emit up to **two** event rows (one per cycle if the cycle's delta is non-zero):

- `ReportingYear=2023, SweepCampaign='2023_big_sweep'` if `2023 Updates != 0`
- `ReportingYear=2026, SweepCampaign='2026_big_sweep'` if `2026 Changes != 0`

Each event gets a matching `QaCorrectionDetail` sidecar with the correction category + rationale.

**Note on PreviousQuantity / NewQuantity**: Ken's XLSX carries deltas only, not absolute before/after counts. The prototype emits `Quantity_Delta` and leaves Previous/New null. Computing absolute values requires joining to `FINAL RES SUMMARY 2012 to 2025.xlsx` Residential sheet — deferred to a follow-up step.

In [5]:
events = []   # parent rows
details = []  # sidecar rows
orphans = []  # validation issues (informational, not dropped)

next_event_id = 1
next_detail_id = 1

CYCLES = [
    # (year_str, sweep, delta_col, rationale_col, category_col, vocab)
    ('2023', '2023_big_sweep', '2023 Updates', '2023 Update Reason', '2023 Summary Reason', VOCAB_2023),
    ('2026', '2026_big_sweep', '2026 Changes', '2026 Detailed',      '2026 Changes Reason', VOCAB_2026),
]

for idx, row in df.iterrows():
    raw_apn = row.get('APN')
    canon = canonical_apn(raw_apn) if isinstance(raw_apn, str) else None
    if canon is None:
        orphans.append({'row_index': idx, 'reason': 'missing_or_invalid_apn',
                        'raw_apn': raw_apn, 'value': None})
        continue

    for cycle_year_str, sweep, delta_col, rationale_col, category_col, vocab in CYCLES:
        delta_val = row.get(delta_col)
        if pd.isna(delta_val) or delta_val == 0:
            continue

        try:
            delta = int(delta_val)
        except (ValueError, TypeError):
            orphans.append({'row_index': idx, 'reason': f'non_numeric_delta_{cycle_year_str}',
                            'raw_apn': raw_apn, 'value': str(delta_val)})
            continue

        category   = row.get(category_col)
        category_str  = str(category).strip()  if isinstance(category, str)  else ''
        rationale  = row.get(rationale_col)
        rationale_str = str(rationale).strip() if isinstance(rationale, str) else ''

        # Soft-validate against the cycle's vocab. Don't drop unknowns —
        # Ken's Sheet1 wording varies from Sheet2's canonical labels.
        # Just flag in orphans for downstream mapping.
        is_canonical = category_str in vocab
        if category_str and not is_canonical:
            orphans.append({'row_index': idx,
                            'reason': f'noncanonical_category_{cycle_year_str}',
                            'raw_apn': raw_apn, 'value': category_str})

        event_id = next_event_id; next_event_id += 1
        events.append({
            'ChangeEventID':      event_id,
            'RawAPN':             raw_apn,
            'CanonicalAPN':       canon,
            'CommodityShortName': 'SFRUU_or_MFRUU_TBD',  # Ken's data is residential; resolve at DB load
            'Year':               int(cycle_year_str),
            'PreviousQuantity':   None,
            'NewQuantity':        None,
            'Quantity_Delta':     delta,
            'ChangeSource':       'qa_correction',
            'Rationale':          rationale_str,
            'EvidenceURL':        None,
            'RecordedBy':         'Ken',
            'RecordedAt':         LOADED_AT,
        })
        details.append({
            'QaCorrectionDetailID':   next_detail_id,
            'ChangeEventID':          event_id,
            'ReportingYear':          int(cycle_year_str),
            'SweepCampaign':          sweep,
            'CorrectionCategory':     category_str,
            'CorrectionCategoryCanonical': is_canonical,  # True if matched vocab
            'SummaryReason':          rationale_str,
            'RawAPN':                 raw_apn,
            'SourceFileSnapshot':     SOURCE_SNAPSHOT,
            'LoadedAt':               LOADED_AT,
        })
        next_detail_id += 1

events_df  = pd.DataFrame(events)
details_df = pd.DataFrame(details)
orphans_df = pd.DataFrame(orphans)

print(f'Events emitted:      {len(events_df):,}')
print(f'Detail rows emitted: {len(details_df):,}')
print(f'Issues flagged:      {len(orphans_df):,}')
if len(details_df):
    pct = details_df['CorrectionCategoryCanonical'].mean() * 100
    print(f'Canonical-vocab match rate: {pct:.1f}%')

Events emitted:      5,925
Detail rows emitted: 5,925
Issues flagged:      4,137
Canonical-vocab match rate: 30.2%


## 6. Validation summary

Quick sanity checks before writing:

- Row counts by `ReportingYear` (should reflect Ken's 2023 vs 2026 sweep volumes)
- Top correction categories
- Sum of `Quantity_Delta` by year (should approximately match Sheet1 cycle totals)
- Orphan reasons distribution

In [6]:
if len(events_df):
    print('=== Events by ReportingYear ===')
    print(events_df.groupby('Year').size().to_string())
    print()
    print('=== Sum of Quantity_Delta by Year ===')
    print(events_df.groupby('Year')['Quantity_Delta'].sum().to_string())
    print()
if len(details_df):
    print('=== Top 10 CorrectionCategory ===')
    print(details_df['CorrectionCategory'].value_counts().head(10).to_string())
    print()
if len(orphans_df):
    print('=== Orphan reasons ===')
    print(orphans_df['reason'].value_counts().to_string())
else:
    print('No orphans — every row passed validation.')

=== Events by ReportingYear ===
Year
2023    3864
2026    2061

=== Sum of Quantity_Delta by Year ===
Year
2023    515
2026     73

=== Top 10 CorrectionCategory ===
CorrectionCategory
Corrections - Units Removed Based on County Data                                        890
Unit(s) not previously counted. Constructed in or before 2012. Verified with County.    733
Correction Based on County Data                                                         696
Unit(s) Not Previously Counted - Added in 2026                                          616
Mobile Home Park Corrections                                                            582
Unit(s) Incorrectly Removed in 2023 - Unit(s) Added Back in 2026                        458
Over-Correction                                                                         349
Additional Unit(s) Removed in 2026                                                      345
Parcel is outside TRPA Boundary                                                

## 7. Write outputs

In [7]:
events_df.to_csv(OUT_EVENTS, index=False)
details_df.to_csv(OUT_DETAIL, index=False)
orphans_df.to_csv(OUT_ORPHANS, index=False)

print(f'Wrote {OUT_EVENTS.name}:  {len(events_df):,} rows')
print(f'Wrote {OUT_DETAIL.name}:  {len(details_df):,} rows')
print(f'Wrote {OUT_ORPHANS.name}: {len(orphans_df):,} rows')

Wrote qa_change_events.csv:  5,925 rows
Wrote qa_correction_detail.csv:  5,925 rows
Wrote qa_load_orphans.csv: 4,137 rows


## Next steps

1. **Reconciliation bridge** — `notebooks/05_qa_reconciliation.ipynb` joins these change events to the s06_qa.py outputs (`QA_Lost_APNs.csv`, `QA_FC_Units_Not_In_CSV.csv`) so we can label each automated detection as `addressed_2023`, `addressed_2026`, `pending`, or `unmatched`.
2. **Compute Previous/New quantities** — join to `from_ken/FINAL RES SUMMARY 2012 to 2025.xlsx` Residential sheet by canonical APN to populate `PreviousQuantity` / `NewQuantity` from absolute year-by-year counts.
3. **Resolve `CommodityShortName`** — Ken's data is residential; once the DB load happens, look up `(SFRUU, MFRUU, ADU)` per parcel from `dbo.ParcelCommodityInventory` or the existing-development snapshot.
4. **Promote to script** — once stable, this notebook becomes `parcel_development_history_etl/steps/s07_load_ca_changes.py`.